# 11 — EfficientNet-B3 training

**Purpose:** Train a two-logit Normal/Blast classifier with a CUDA smoke test and head/fine phases.

**Required inputs:** Saved subject splits; CUDA-enabled PyTorch; ImageNet weights download or cache.

**Saved outputs:** Timestamped outputs/classifier/runs/<run>/ checkpoints, logs, CSV and plots.

## 1. Load central configuration

Set `cnmc_root` once in ignored `config.local.yaml`. Source images are never moved or changed.

In [ ]:
from pathlib import Path
import os, sys
search = [Path.cwd(), *Path.cwd().parents]
if os.environ.get("LEUKEMIA_YOLO_ROOT"):
    search.insert(0, Path(os.environ["LEUKEMIA_YOLO_ROOT"]))
for candidate in search:
    if (candidate / "config.example.yaml").is_file() and (candidate / "src" / "classifier_config.py").is_file():
        sys.path.insert(0, str(candidate)); break
from src.classifier_config import classifier_config, classifier_output
cfg = classifier_config()
out = classifier_output(cfg)
print("Project:", cfg["root"])
print("Config:", cfg["config_path"])
print("C-NMC source:", cfg["cnmc_root"])

## A. GPU and data readiness

GPU training never falls back silently to CPU.

In [ ]:
from src.classifier_training import require_classifier_cuda, class_weights
from src.cnmc_splits import create_or_load_splits, validate_splits
import torch
device = require_classifier_cuda()
splits, split_info = create_or_load_splits(cfg)
display(__import__("pandas").DataFrame(validate_splits(splits)).T)
print("GPU:", torch.cuda.get_device_name(0), "free GiB:", round(torch.cuda.mem_get_info(0)[0]/2**30, 2))
print("Image size:", cfg["classifier_image_size"], "batch:", cfg["classifier_batch_size"])
print("Class weights [Normal, Blast]:", class_weights(splits["train"], device).tolist())

## B. Model architecture

Torchvision's DEFAULT ImageNet B3 weights; final head returns exactly two logits.

In [ ]:
from src.classifier_model import build_model, set_trainable, metadata
model = build_model(pretrained=True).to(device)
set_trainable(model, "head")
print(model.classifier)
print(metadata(cfg["classifier_image_size"]))

## C. Preview the deterministic validation batch

At 224 pixels this differs from B3's original 300-pixel center-crop recipe; the choice is recorded.

In [ ]:
from src.classifier_transforms import CNMCDataset
from torch.utils.data import DataLoader
batch = next(iter(DataLoader(CNMCDataset(splits["val"], cfg["classifier_image_size"]), batch_size=2)))
print("Batch RGB tensor:", batch[0].shape, "labels:", batch[1].tolist())
assert batch[0].shape[1:] == (3, cfg["classifier_image_size"], cfg["classifier_image_size"])

## D. One-batch forward and backward check

Only the new head receives gradients in phase A.

In [ ]:
from torch import nn
model.train(); model.features.eval()
x, y = batch[0].to(device), batch[1].to(device)
logits = model(x); assert logits.shape == (2, 2)
loss = nn.CrossEntropyLoss()(logits, y); loss.backward()
assert model.classifier[-1].weight.grad is not None
print("PASS — forward/backward; loss:", float(loss.detach()))
model.zero_grad(set_to_none=True); del x, y, logits, loss

## E. Launch one-epoch smoke test

The guarded child script handles Windows workers. Logs stream live and are saved in its run folder.

In [ ]:
import subprocess, sys
def stream_command(arguments):
    command = [sys.executable, "-u", *map(str, arguments)]
    process = subprocess.Popen(command, cwd=cfg["root"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, encoding="utf-8", errors="replace", bufsize=1)
    try:
        for line in process.stdout: print(line, end="", flush=True)
    except KeyboardInterrupt:
        process.terminate(); process.wait()
        raise RuntimeError("Training stopped. Resume from the run's last_checkpoint.pth")
    code = process.wait()
    if code: raise RuntimeError(f"Child process exited {code}; inspect training_log.txt")
stream_command([cfg["root"] / "scripts" / "train_classifier.py", "--mode", "smoke"])

## F. Verify smoke artifacts

Full training is gated on a successful CUDA checkpoint.

In [ ]:
from src.classifier_model import load_checkpoint
smokes = sorted((out / "runs").glob("smoke_*/best_model.pth"))
assert smokes, "No successful CUDA smoke checkpoint"
smoke_model, smoke_info, _ = load_checkpoint(smokes[-1], device)
print("PASS — loaded smoke checkpoint:", smokes[-1])
del smoke_model

## G. Run full two-phase training

Separate timestamped run. Stop with Interrupt; resume with --resume last_checkpoint.pth.

In [ ]:
stream_command([cfg["root"] / "scripts" / "train_classifier.py", "--mode", "full"])
print("PASS — child process completed")

## H. Review outputs and GPU memory

History is written after each complete epoch; incomplete CSV lines are ignored by a fresh read after exit.

In [ ]:
import pandas as pd
from src.classifier_config import latest_classifier_run
run = latest_classifier_run(cfg)
history = pd.read_csv(run / "training_history.csv")
display(history.tail(10))
print("Child-process peak allocated VRAM GiB:", round(history.peak_vram_bytes.max()/2**30, 2))
print("Training log:", run / "training_log.txt")

## I. Inspect curves and selected checkpoint

The best model is selected by validation macro-F1, never test metrics.

In [ ]:
from IPython.display import Image as DisplayImage, display
for filename in ("loss_curve.png", "accuracy_curve.png", "f1_curve.png"):
    display(DisplayImage(filename=str(run / filename)))
best_model, best_info, selected = load_checkpoint(run / "best_model.pth", device)
print("PASS — best checkpoint:", run / "best_model.pth")
print("Validation macro-F1:", selected["validation_macro_f1"], "class mapping:", best_info["class_mapping"])

## Next

Open **12_EfficientNet_B3_Evaluation.ipynb**. Test fold_2 is used only after model selection.